# LLM Adaptation Workflow
### Notebook 02 — Data Preparation

---

## The central challenge

A foundation model knows a lot about the world in general. But to be useful to a specific organisation, it needs to know **their** things:
- Their products, clients, and internal terminology
- Specific documents: contracts, reports, policies, research
- Domain conventions: how a financial analyst writes, how a lawyer structures arguments

The raw materials are usually **unstructured documents** — PDFs, HTML pages, plain text files. We can't feed these directly into a training loop. We need to transform them into a format the model can learn from.

This notebook covers:
1. **Downloading public financial documents** (SEC EDGAR filings)
2. **Cleaning and chunking** — preparing raw text for downstream use
3. **Building an instruction dataset** — the format used for fine-tuning
4. **Synthetic data generation** — using an LLM to create more training examples

---

## 🔧 Adapt This

We use SEC filings as our public corpus. To adapt this to your domain:
- **PDFs**: use `pypdf` to extract text, then proceed from the cleaning step
- **Internal wikis / Confluence**: export to markdown, load with `pathlib`
- **Web pages**: use `trafilatura` or `BeautifulSoup` to extract clean text
- **Database records**: query to a pandas DataFrame, convert rows to text with a template

Everything after loading is identical regardless of source.

In [ ]:
import os
import re
import json
import random
import textwrap
from pathlib import Path

import pandas as pd
from datasets import Dataset

# Set seeds for reproducibility — important when generating datasets
random.seed(42)

# Paths — all data goes into the data/ directory at the project root
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Directories ready.")

---

## 1. Downloading SEC filings from EDGAR

The SEC (U.S. Securities and Exchange Commission) requires public companies to file regular reports. These are freely available via the EDGAR database and contain detailed financial information.

The key filings we care about:
- **10-K** — annual report (comprehensive financial results, risk factors, business overview)
- **10-Q** — quarterly update
- **8-K** — material events (earnings announcements, major news)

These are rich, structured documents — ideal for building a financial domain corpus.

In [ ]:
# We use the sec-edgar-downloader library to pull filings programmatically.
# SEC requires a user agent string (your name/email) — this is a public API.

from sec_edgar_downloader import Downloader

# Initialise downloader
# Replace with your own name/email (SEC requirement, no authentication needed)
dl = Downloader("LLM Adaptation Project", "research@example.com", str(RAW_DIR))

# Download 10-K filings for a few large public companies
# limit=2 means we get the 2 most recent annual reports per company
companies = {
    "AAPL": "Apple",
    "MSFT": "Microsoft",
    "JPM": "JPMorgan Chase",
}

for ticker, name in companies.items():
    print(f"Downloading {name} ({ticker}) annual report...")
    dl.get("10-K", ticker, limit=2)

print("\nDownload complete. Files saved to:", RAW_DIR)

In [ ]:
# Let's see what we downloaded
import os

all_files = list(RAW_DIR.rglob("*.txt")) + list(RAW_DIR.rglob("*.htm"))
print(f"Found {len(all_files)} files")
for f in all_files[:10]:  # show first 10
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:50s}  {size_kb:8.0f} KB")

---

## 2. Cleaning raw text

SEC filings are HTML or plain text and contain a lot of noise: HTML tags, legal boilerplate, repeated headers, page numbers, and whitespace. Before we use this text for anything, we need to clean it.

In [ ]:
def clean_text(raw_text: str) -> str:
    """
    Clean raw SEC filing text.
    Steps:
      1. Strip HTML tags if present
      2. Normalise whitespace
      3. Remove very short lines (page numbers, stray chars)
      4. Collapse repeated blank lines
    """
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', raw_text)
    
    # Decode common HTML entities
    text = text.replace('&amp;', '&').replace('&nbsp;', ' ').replace('&lt;', '<').replace('&gt;', '>')
    
    # Normalise whitespace within lines
    lines = text.split('\n')
    lines = [' '.join(line.split()) for line in lines]
    
    # Drop lines that are too short to be meaningful (e.g. page numbers, lone symbols)
    lines = [line for line in lines if len(line) > 40]
    
    # Rejoin
    text = '\n'.join(lines)
    
    # Collapse multiple blank lines into one
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    return text.strip()


# Load and clean one filing as a demo
sample_file = next(RAW_DIR.rglob("*.txt"), None) or next(RAW_DIR.rglob("*.htm"), None)

if sample_file:
    raw = sample_file.read_text(encoding="utf-8", errors="ignore")
    cleaned = clean_text(raw)
    
    print(f"File: {sample_file.name}")
    print(f"Raw length   : {len(raw):,} characters")
    print(f"Cleaned length: {len(cleaned):,} characters")
    print(f"Reduction    : {(1 - len(cleaned)/len(raw))*100:.0f}%")
    print()
    print("--- First 500 characters of cleaned text ---")
    print(cleaned[:500])
else:
    print("No files found — check the download step above.")

---

## 3. Chunking

A 10-K filing can be hundreds of pages long. Models have a **context window** — a maximum number of tokens they can see at once. We need to split documents into chunks small enough to fit.

Two things to think about when chunking:
- **Chunk size**: too small = loses context, too large = doesn't fit in the window
- **Overlap**: a small overlap between consecutive chunks prevents information being lost at boundaries

In [ ]:
def chunk_text(text: str, chunk_size: int = 512, overlap: int = 64) -> list[str]:
    """
    Split text into overlapping chunks by word count.
    
    chunk_size: target number of words per chunk
    overlap: number of words shared between consecutive chunks
    """
    words = text.split()
    chunks = []
    start = 0
    
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = ' '.join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap  # move forward by (chunk_size - overlap)
    
    return chunks


# Apply to our cleaned text
if sample_file:
    chunks = chunk_text(cleaned, chunk_size=512, overlap=64)
    
    print(f"Document split into {len(chunks)} chunks")
    print(f"Average chunk size: {sum(len(c.split()) for c in chunks) / len(chunks):.0f} words")
    print()
    print("--- Example chunk ---")
    print(textwrap.fill(chunks[5], width=80))

---

## 4. Building an instruction dataset

Fine-tuning requires data in a specific format: **instruction–response pairs**.

Instead of raw text, the model learns from examples like:

```
### Instruction:
What were Apple's primary sources of revenue in the most recent fiscal year?

### Context:
Apple reported total net sales of $383.3 billion for fiscal 2023. Products revenue
was $298.1 billion, driven by iPhone at $200.6 billion...

### Response:
Apple's primary revenue sources were Products ($298.1B) and Services ($85.2B)...
```

This format teaches the model to **follow instructions** and **ground answers in provided context** — which is exactly what we want for a document Q&A assistant.

In [ ]:
import sys
sys.path.append("..")
from utils.templates import format_instruction

# For demonstration, we'll create a small handcrafted dataset based on real SEC data.
# In production, you'd generate these at scale using the synthetic approach in the next section.

manual_examples = [
    {
        "instruction": "What was Apple's total net sales for fiscal year 2023?",
        "context": "Apple Inc. reported total net sales of $383.3 billion for the fiscal year ended September 30, 2023, compared to $394.3 billion for fiscal 2022, representing a decrease of 2.8%.",
        "response": "Apple's total net sales for fiscal year 2023 were $383.3 billion, a decrease of 2.8% from $394.3 billion in fiscal 2022."
    },
    {
        "instruction": "What are the main risks Apple identifies in its annual report?",
        "context": "The Company's operations and financial results are subject to various risks and uncertainties, including global economic conditions, intense competition, rapid technological change, dependence on component suppliers, and cybersecurity threats.",
        "response": "Apple identifies several key risks in its annual report: global economic conditions affecting consumer spending, intense competition in its product markets, rapid technological change requiring continuous innovation, supplier concentration risk, and cybersecurity and data privacy threats."
    },
    {
        "instruction": "How does JPMorgan Chase describe its credit risk management approach?",
        "context": "JPMorgan Chase manages credit risk through diversification, credit limits, collateral requirements, and credit derivatives. The firm uses internal credit ratings and stress testing to assess portfolio quality.",
        "response": "JPMorgan Chase manages credit risk through four main mechanisms: portfolio diversification to avoid concentration, credit exposure limits per counterparty, collateral requirements to secure exposures, and hedging via credit derivatives. The firm supplements these controls with internal credit ratings and stress testing."
    },
    {
        "instruction": "What does Microsoft report as its three main business segments?",
        "context": "Microsoft operates through three segments: Productivity and Business Processes (including Office and LinkedIn), Intelligent Cloud (including Azure and server products), and More Personal Computing (including Windows and Xbox).",
        "response": "Microsoft's three business segments are: (1) Productivity and Business Processes, which includes Office 365, LinkedIn, and Dynamics; (2) Intelligent Cloud, anchored by Azure cloud services and server products; and (3) More Personal Computing, covering Windows, Xbox, and Surface devices."
    },
    {
        "instruction": "Summarise the key financial performance metrics for Microsoft in fiscal 2023.",
        "context": "Microsoft reported revenue of $211.9 billion for fiscal year 2023, up 7% year-over-year. Operating income was $88.5 billion (+6%), and net income was $72.4 billion (+0.4%). Azure and cloud services grew 27%.",
        "response": "Microsoft's fiscal 2023 financial highlights: revenue of $211.9B (+7% YoY), operating income of $88.5B (+6%), and net income of $72.4B (roughly flat). The standout performer was Azure and cloud services, which grew 27% and demonstrates the ongoing shift to Microsoft's cloud business."
    },
]

# Format each example using the instruction template
formatted = [
    format_instruction(ex["instruction"], ex["response"], ex["context"])
    for ex in manual_examples
]

print("Example formatted instruction:")
print("-" * 60)
print(formatted[0])

---

## 5. Synthetic data generation

Hand-labelling enough Q&A pairs for fine-tuning takes weeks. A much faster approach: **use an LLM to generate training data**.

The idea:
1. Take a chunk of your document
2. Ask a capable LLM to generate a question and answer based on that chunk
3. Use the resulting pairs as training data for your smaller, fine-tuned model

This is sometimes called **LLM-as-teacher** or **self-instruct**. It scales well — you can generate thousands of examples from any document corpus.

> **Note:** For synthetic generation in production, you'd use a strong model like Claude or GPT-4 via API. Here we demonstrate the pattern using the local model so no API key is needed.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokeniser = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

# Build a text generation pipeline — a convenient wrapper around tokenise → generate → decode
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokeniser,
    max_new_tokens=150,
    do_sample=True,
    temperature=0.8,
)

print("Model ready for synthetic data generation.")

In [ ]:
def generate_qa_from_chunk(chunk: str, generator) -> dict | None:
    """
    Given a document chunk, ask the model to generate a question and answer.
    Returns a dict with keys: instruction, context, response
    """
    prompt = f"""Based on the following financial text, generate one specific question 
and a concise answer. Format your response as:
Q: [question]
A: [answer]

Text: {chunk[:400]}

Q:"""
    
    output = generator(prompt)[0]["generated_text"]
    generated = output[len(prompt):].strip()
    
    # Parse out Q and A from the generated text
    if "\nA:" in generated:
        parts = generated.split("\nA:", 1)
        question = parts[0].strip().lstrip("Q:").strip()
        answer = parts[1].strip().split("\n")[0].strip()
        
        if len(question) > 20 and len(answer) > 20:
            return {
                "instruction": question,
                "context": chunk[:400],
                "response": answer,
            }
    return None


# Generate a few synthetic examples from our document chunks
synthetic_examples = []
example_chunks = chunks[:5] if 'chunks' in dir() else []

if example_chunks:
    print("Generating synthetic Q&A pairs from document chunks...\n")
    for i, chunk in enumerate(example_chunks):
        result = generate_qa_from_chunk(chunk, generator)
        if result:
            synthetic_examples.append(result)
            print(f"Example {i+1}:")
            print(f"  Q: {result['instruction']}")
            print(f"  A: {result['response'][:100]}...")
            print()
    print(f"Generated {len(synthetic_examples)} valid examples from {len(example_chunks)} chunks.")
else:
    print("No chunks available — using the manual examples from earlier.")
    synthetic_examples = manual_examples

---

## 6. Saving the dataset

We combine our manual and synthetic examples into a HuggingFace `Dataset` — the standard format used by the training libraries throughout this workflow.

In [ ]:
# Combine manual and synthetic examples
all_examples = manual_examples + synthetic_examples

# Add formatted text column (the instruction template applied to each example)
for ex in all_examples:
    ex["text"] = format_instruction(ex["instruction"], ex["response"], ex.get("context"))

# Convert to HuggingFace Dataset
dataset = Dataset.from_list(all_examples)

# Split into train / validation
# 90% train, 10% validation — standard practice
split = dataset.train_test_split(test_size=0.1, seed=42)

print(f"Total examples : {len(dataset)}")
print(f"Train split    : {len(split['train'])}")
print(f"Val split      : {len(split['test'])}")
print()
print("Dataset schema:")
print(dataset.features)

In [ ]:
# Save to disk in a format compatible with the rest of the workflow
split.save_to_disk(str(PROCESSED_DIR / "instruction_dataset"))

# Also save a JSON version for easy inspection
with open(PROCESSED_DIR / "examples.json", "w") as f:
    json.dump(all_examples, f, indent=2)

print(f"Dataset saved to {PROCESSED_DIR}")
print()
print("Files created:")
for p in PROCESSED_DIR.rglob("*"):
    if p.is_file():
        print(f"  {p.relative_to(PROCESSED_DIR)}")

---

## Summary

In this notebook we:
- Downloaded real public financial filings from the SEC EDGAR database
- Cleaned raw HTML/text to remove noise
- Chunked long documents into model-friendly pieces with overlap
- Built an instruction dataset (the format needed for fine-tuning)
- Used an LLM to generate synthetic Q&A pairs at scale — the same technique used in modern dataset construction pipelines
- Saved everything in HuggingFace Dataset format

With this dataset, we can now take two paths:
- **Notebook 03**: Build a RAG system using the raw chunks as a retrieval corpus
- **Notebook 04**: Fine-tune the model on the instruction dataset

---

▶ **Next: [Notebook 03 — RAG Pipeline](03_rag_pipeline.ipynb)**